In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
%sql
ALTER TABLE dbw_agentic_ai_dev.telco_ai.customer_note_embeddings
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

In [0]:
# Step 1: import Vector Search client
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()


#Step 2: Define names
VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"
SOURCE_TABLE = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings"
INDEX_NAME = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"


embedding_dimension = len(
    spark.table(SOURCE_TABLE)
         .select("embedding")
         .first()["embedding"]
)


##### Create Vector Search Endpoint

In [0]:
# -----------------------------
# Create Vector Search client
# -----------------------------
vsc = VectorSearchClient()


# -----------------------------
# Create Vector Search Endpoint
# If it already exists, continue
# -----------------------------
try:
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    print("Vector Search endpoint created.")
except Exception as e:
    print("Vector Search endpoint may already exist.")
    print(e)


##### Create Delta Sync Index

In [0]:
# -----------------------------
# Create Delta Sync Index
# If it already exists, continue
# -----------------------------


try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        source_table_name=SOURCE_TABLE,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="customer_id",
        embedding_dimension=embedding_dimension,
        embedding_vector_column="embedding",
        columns_to_sync=[
           "customer_id", "note"
        ]
    )
    print("Delta Sync Index created.")
except Exception as e:
    print("Delta Sync Index may already exist.")
    print(e)

    index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=INDEX_NAME
    )

#vsc.delete_index(
    #endpoint_name="telco-vector-search-endpoint",
    #index_name=INDEX_NAME
#)

# -----------------------------
# Synchronize the Delta Sync Index with the latest
# changes from the Delta table.
# -----------------------------
index.sync()
print("Index sync triggered.")


In [0]:
vsc.delete_index(
    endpoint_name="telco-vector-search-endpoint",
    index_name=INDEX_NAME
)